Рассчет метрик

In [ ]:
import re
import pandas as pd
from typing import List, Tuple, Dict, Optional

In [ ]:
DOTS = {'.'}
COMMAS = {','}
OTHERS = {'!', '?', ':', ';', '…', '—', '–', '-'}

GROUP_ORDER = ['dots', 'commas', 'others', 'capital']
GROUP_NAMES = {
    'dots': 'Точки (.)',
    'commas': 'Запятые (,)',
    'others': 'Прочие (! ? : ; … — – и др.)',
    'capital': 'Заглавные буквы',
}

def normalize_text_for_metrics(text: str) -> str:
    """Нормализация текста: заменяет ё на е, убирает лишние пробелы"""
    if not isinstance(text, str):
        return ''
    text = text.replace('ё', 'е').replace('Ё', 'Е')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_word(w: str) -> str:
    """Нормализация слова: нижний регистр, заменяет ё на е, удаление лишних символов (кроме дефисов и цифр)"""
    if not w:
        return ''
    w = w.lower().replace('ё', 'е')
    w = re.sub(r'[^\w\-]+', '', w, flags=re.UNICODE)
    return w


def levenshtein(a: str, b: str) -> int:
    """Расстояние Левенштейна между словами a и b"""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    # динамиечски заполняет таблицу, где слово a горизонтали, а b - по вертикали
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            curr[j] = min(
                prev[j] + 1,       # удаление ca
                curr[j - 1] + 1,   # вставка cb
                prev[j - 1] + cost # замена
            )
        prev = curr # на следующую строку таблицы
    return prev[-1] # ответ - в нижней правой клетке


def words_similar(w1: str, w2: str,
                  max_dist_short: int = 1,
                  max_dist_long: int = 2,
                  ratio: float = 0.3) -> bool:
    """
    Гибридный критерий похожести слов:
    - для коротких слов (≤4 символов) — абсолютный порог,
    - для длинных — по Левенштейну.
    """
    if w1 == w2:
        return True
    if not w1 or not w2:
        return False
    d = levenshtein(w1, w2)
    if len(w1) <= 4 or len(w2) <= 4:
        return d <= max_dist_short
    return d / max(len(w1), len(w2)) <= ratio


def extract_pairs(text: str) -> List[Tuple[str, str, str]]:
    """
    Возвращает список троек (L, R, punct), где:
      L — нормализованное слово слева от знака (или '' в начале),
      R — нормализованное слово справа от знака (или '' в конце),
      punct — строка знаков между L и R.
    """
    text = normalize_text_for_metrics(text)

    # Токены: слова (с дефисами) или группы знаков
    tokens = re.findall(r'[\w\-]+|[.!?,;:…—–\-]+', text, re.UNICODE)

    # Разделяем на слова и знаки
    items = [] # список кортежей (слово/пункт, str)
    for t in tokens:
        if re.match(r'[\w\-]+', t, re.UNICODE):
            items.append(('word', normalize_word(t)))
        else:
            items.append(('punct', t))

    pairs = []
    prev_word = ''  # слово слева от текущей позиции

    i = 0
    while i < len(items):
        kind, val = items[i]
        if kind == 'word':
            # Собираем знаки после слова
            punct = ''
            j = i + 1
            while j < len(items) and items[j][0] == 'punct': # собираем все знаки после текущего слова
                punct += items[j][1]
                j += 1
            # Правое слово — следующее слово, если оно есть
            if j < len(items) and items[j][0] == 'word':
                right_word = items[j][1]
            else:
                right_word = ''
            pairs.append((prev_word, right_word, punct))
            prev_word = val
            i = j
        else:
            # Текст начинается со знака (например, - Привет. ) - > (ничего, знак, слово справа)
            punct = val
            j = i + 1
            while j < len(items) and items[j][0] == 'punct':
                punct += items[j][1]
                j += 1
            right_word = items[j][1] if j < len(items) and items[j][0] == 'word' else ''
            pairs.append(('', right_word, punct))
            i = j

    return pairs


def extract_word_triples(text: str) -> List[Tuple[str, str, str]]:
    """
    Возвращает список троек (prev, current, next) нормализованных слов
    в исходном регистре для оценки заглавных букв.
    prev/next могут быть '' на границах.
    """
    text = normalize_text_for_metrics(text)
    # Ищем слова с сохранением регистра
    words = re.findall(r'[\w\-]+', text, re.UNICODE)
    triples = []
    for i, w in enumerate(words):
        prev = words[i - 1] if i > 0 else ''
        nxt = words[i + 1] if i + 1 < len(words) else ''
        triples.append((prev, w, nxt))
    return triples


def classify_punct(p: str) -> str:
    """
    Возвращает 'dots', 'commas', 'others' или 'none' для строки знаков.
    Приоритет: если есть хоть одна точка и ничего кроме — dots;
    если есть запятая и ничего кроме — commas; иначе others.
    """
    if not p:
        return 'none'
    chars = set(p)
    if chars == DOTS:
        return 'dots'
    if chars == COMMAS:
        return 'commas'
    return 'others'


def align_pairs(ref_pairs, hyp_pairs, max_lookahead: int = 10):
    """
    Идёт по эталонным парам. Для каждой ищет соответствие в гипотезе,
    начиная с текущей позиции, в пределах max_lookahead.
    Останавливается при первом совпадении или при "наткнулись на следующую пару".
    Возвращает список кортежей (ref_pair, hyp_pair_or_None, reason).
    reason ∈ {'found', 'missed_by_next', 'missed_by_end'}
    """
    aligned = []
    i_h = 0 #  указатель на текущую позицию в гипотезе, чтобы не смотреть назад
    n_h = len(hyp_pairs)
    n_r = len(ref_pairs)

    for i_r, ref_pair in enumerate(ref_pairs):
        L_e, R_e, p_e = ref_pair
        # Следующая эталонная пара (для проверки, что не наткнулись на нее, ища текущую)
        next_ref = ref_pairs[i_r + 1] if i_r + 1 < n_r else None

        found = None # индекс найденной пары в гипотезе
        hit_next = None # индекс пары гипотезы, которая соответствует следующей эталонной

        j = i_h
        limit = min(n_h, i_h + max_lookahead) # ищем от текущего индекса до +10 пар вправо
        while j < limit:
            L_h, R_h, p_h = hyp_pairs[j]
            if words_similar(L_h, L_e) and words_similar(R_h, R_e):
                found = j
                break
            if next_ref is not None:
                Ln, Rn, _ = next_ref
                if words_similar(L_h, Ln) and words_similar(R_h, Rn):
                    hit_next = j
                    break
            j += 1

        if found is not None: # нашли 
            aligned.append((ref_pair, hyp_pairs[found], 'found'))
            i_h = found + 1
        elif hit_next is not None: # наткнулись на следующую пару, следующую будем искать начиная с этой же позиции
            aligned.append((ref_pair, None, 'missed_by_next'))
            i_h = hit_next
        else: # ничего не нашли
            aligned.append((ref_pair, None, 'missed_by_end'))
            # i_h не двигаем — следующая итерация может найти совпадение дальше

    return aligned



def compute_metrics_v2(ref_text: str, hyp_text: str,
                       debug_log: Optional[List[Dict]] = None,
                       model_name: str = '') -> Dict[str, Dict]:
  
    ref_pairs = extract_pairs(ref_text)
    hyp_pairs = extract_pairs(hyp_text)
    aligned = align_pairs(ref_pairs, hyp_pairs)

    stats = {g: {'tp': 0, 'fp': 0, 'fn': 0,
                 'visibility_hit': 0,
                 'total_ref': 0, 'total_hyp': 0}
             for g in ['dots', 'commas', 'others']}

    for idx, (ref_pair, hyp_pair, reason) in enumerate(aligned):
        L_e, R_e, p_e = ref_pair 
        ref_cls = classify_punct(p_e) # класс эталонного знака

        if reason == 'found':
            _, _, p_h = hyp_pair
            hyp_cls = classify_punct(p_h)
        else:
            p_h = ''
            hyp_cls = 'none'

        outcome_per_group = {}
        for g in ['dots', 'commas', 'others']: # метрики отдельно для каждой группы знаков
            ref_has = (ref_cls == g)
            hyp_has = (hyp_cls == g)

            if ref_has and hyp_has:
                outcome_per_group[g] = 'tp'
            elif ref_has and not hyp_has:
                outcome_per_group[g] = 'fn'
            elif hyp_has and not ref_has:
                outcome_per_group[g] = 'fp'
            else:
                outcome_per_group[g] = '-'

            if ref_has:
                stats[g]['total_ref'] += 1
                if hyp_has:
                    stats[g]['tp'] += 1
                    stats[g]['visibility_hit'] += 1
                else:
                    stats[g]['fn'] += 1
                    # visibility: был ли поставлен хоть какой-то знак
                    if p_h:
                        stats[g]['visibility_hit'] += 1

            if hyp_has and not ref_has:
                stats[g]['fp'] += 1
                stats[g]['total_hyp'] += 1
            elif hyp_has:
                stats[g]['total_hyp'] += 1

        # логирование
        if debug_log is not None:
            debug_log.append({
                'model': model_name,
                'pair_idx': idx,
                'L_ref': L_e,
                'R_ref': R_e,
                'punct_ref': p_e,
                'class_ref': ref_cls,
                'L_hyp': hyp_pair[0] if hyp_pair else '',
                'R_hyp': hyp_pair[1] if hyp_pair else '',
                'punct_hyp': p_h,
                'class_hyp': hyp_cls,
                'reason': reason,
                'outcome_dots':   outcome_per_group['dots'],
                'outcome_commas': outcome_per_group['commas'],
                'outcome_others': outcome_per_group['others'],
            })

    def finalize(s): # расчет самих метрик по формулам
        precision = s['tp'] / (s['tp'] + s['fp']) if (s['tp'] + s['fp']) else 0.0
        recall = s['tp'] / s['total_ref'] if s['total_ref'] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        visibility = s['visibility_hit'] / s['total_ref'] if s['total_ref'] else 0.0
        return {
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'f1': round(f1, 4),
            'visibility': round(visibility, 4),
            'tp': s['tp'],
            'fp': s['fp'],
            'fn': s['fn'],
            'extra': s['fp'],
            'total_ref': s['total_ref'],
            'total_hyp': s['total_hyp'],
        }

    result = {g: finalize(stats[g]) for g in ['dots', 'commas', 'others']}


    # для регистра
    ref_triples = extract_word_triples(ref_text)
    hyp_triples = extract_word_triples(hyp_text)

    # Сопоставляем тройки по центральному слову (по Левенштейну)
    cap_stats = {'tp': 0, 'fp': 0, 'fn': 0,
                 'total_ref': 0, 'total_hyp': 0}

    i_h = 0
    n_h = len(hyp_triples)
    max_lookahead = 10

    for i_r, (prev_e, cur_e, next_e) in enumerate(ref_triples):
        cur_norm = normalize_word(cur_e)
        prev_norm = normalize_word(prev_e)
        next_norm = normalize_word(next_e)

        next_ref = ref_triples[i_r + 1] if i_r + 1 < len(ref_triples) else None
        next_cur_norm = normalize_word(next_ref[1]) if next_ref else None

        found = None
        j = i_h
        limit = min(n_h, i_h + max_lookahead)
        while j < limit:
            _, cur_h, _ = hyp_triples[j]
            cur_h_norm = normalize_word(cur_h)
            if words_similar(cur_h_norm, cur_norm):
                found = j
                break
            if next_cur_norm and words_similar(cur_h_norm, next_cur_norm):
                break
            j += 1

        ref_is_cap = cur_e[0].isupper() if cur_e else False

        if found is not None:
            _, cur_h, _ = hyp_triples[found]
            hyp_is_cap = cur_h[0].isupper() if cur_h else False
            if ref_is_cap:
                cap_stats['total_ref'] += 1
                if hyp_is_cap:
                    cap_stats['tp'] += 1
                else:
                    cap_stats['fn'] += 1
            if hyp_is_cap and not ref_is_cap:
                cap_stats['fp'] += 1
                cap_stats['total_hyp'] += 1
            elif hyp_is_cap:
                cap_stats['total_hyp'] += 1
            i_h = found + 1
        else:
            # не нашли — считаем fn для ref, если там была заглавная
            if ref_is_cap:
                cap_stats['total_ref'] += 1
                cap_stats['fn'] += 1
            # i_h не двигаем

    # метрики по формулам
    cap_precision = cap_stats['tp'] / (cap_stats['tp'] + cap_stats['fp']) if (cap_stats['tp'] + cap_stats['fp']) else 0.0
    cap_recall = cap_stats['tp'] / cap_stats['total_ref'] if cap_stats['total_ref'] else 0.0
    cap_f1 = 2 * cap_precision * cap_recall / (cap_precision + cap_recall) if (cap_precision + cap_recall) else 0.0

    result['capital'] = {
        'precision': round(cap_precision, 4),
        'recall': round(cap_recall, 4),
        'f1': round(cap_f1, 4),
        'visibility': round(cap_recall, 4),  # для регистра visibility = recall
        'tp': cap_stats['tp'],
        'fp': cap_stats['fp'],
        'fn': cap_stats['fn'],
        'extra': cap_stats['fp'],
        'total_ref': cap_stats['total_ref'],
        'total_hyp': cap_stats['total_hyp'],
    }

    return result


def evaluate_all_models(reference_path: str, excel_path: str,
                        output_path: str = 'final_metrics_second.xlsx'):

    with open(reference_path, 'r', encoding='utf-8') as f:
        reference_text = f.read()

    df = pd.read_excel(excel_path)

    model_columns = [c for c in df.columns if c.endswith('_result')]
    rows = []
    debug_log = []  # общий список для всех моделей

    for col in model_columns:
        model_name = col.replace('_result', '')
        hyp_text = ' '.join(df[col].dropna().astype(str))

        metrics = compute_metrics_v2(
            reference_text, hyp_text,
            debug_log=debug_log,
            model_name=model_name,
        )

        for group, vals in metrics.items():
            row = {
                'model': model_name,
                'group': group,
                'group_name': GROUP_NAMES.get(group, group),
            }
            row.update(vals)
            rows.append(row)

    df_metrics = pd.DataFrame(rows)
    df_log = pd.DataFrame(debug_log)

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_metrics.to_excel(writer, sheet_name='metrics', index=False)
        df_log.to_excel(writer, sheet_name='debug_log', index=False)

    return df_metrics, df_log

In [ ]:
REFERENCE_PATH = 'reference_text_2.txt'
EXCEL_PATH = 'final_second.xlsx'

df_metrics = evaluate_all_models(REFERENCE_PATH, EXCEL_PATH)